# Refrigerated Vehicles Problem in Pure Python
## Five Exact Methods for Book Problem 2.4


In [16]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations, product
from math import ceil, floor, isclose
from time import perf_counter


In [17]:
EPSILON = 1e-9


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def dot_product(vector_a, vector_b):
    return sum(a * b for a, b in zip(vector_a, vector_b))


def is_integer_value(value: float) -> bool:
    return isclose(value, round(value), abs_tol=EPSILON)


def relation_holds(lhs: float, relation: str, rhs: float) -> bool:
    if relation == "<=":
        return lhs <= rhs + EPSILON
    if relation == ">=":
        return lhs >= rhs - EPSILON
    if relation == "=":
        return isclose(lhs, rhs, abs_tol=EPSILON)
    raise ValueError(f"Unsupported relation: {relation}")


def objective_value(problem, values):
    return dot_product(problem["objective"], values)


def solution_from_values(problem, values):
    objective = int(round(objective_value(problem, values)))
    solution = {
        name: int(value)
        for name, value in zip(problem["variable_names"], values)
    }
    solution[problem["objective_key"]] = objective
    return solution


def is_feasible(problem, values):
    for coefficients, relation, rhs in problem["constraints"]:
        lhs = dot_product(coefficients, values)
        if not relation_holds(lhs, relation, rhs):
            return False
    return True


def compare_values(problem, left_values, right_values):
    if left_values is None and right_values is None:
        return 0
    if left_values is None:
        return -1
    if right_values is None:
        return 1

    left_objective = objective_value(problem, left_values)
    right_objective = objective_value(problem, right_values)

    if problem["sense"] == "max":
        if left_objective > right_objective + EPSILON:
            return 1
        if left_objective < right_objective - EPSILON:
            return -1
    else:
        if left_objective < right_objective - EPSILON:
            return 1
        if left_objective > right_objective + EPSILON:
            return -1

    left_key = tuple(int(value) for value in left_values)
    right_key = tuple(int(value) for value in right_values)
    if left_key < right_key:
        return 1
    if left_key > right_key:
        return -1
    return 0


def choose_better(problem, candidate_values, best_values):
    return candidate_values if compare_values(problem, candidate_values, best_values) > 0 else best_values


In [18]:
def feasible_interval_for_y(problem, x_value):
    y_lower, y_upper = problem["bounds"][1]

    for coefficients, relation, rhs in problem["constraints"]:
        coefficient_x, coefficient_y = coefficients
        constant = rhs - coefficient_x * x_value

        if abs(coefficient_y) < EPSILON:
            if not relation_holds(coefficient_x * x_value, relation, rhs):
                return None
            continue

        boundary = constant / coefficient_y

        if relation == "<=":
            if coefficient_y > 0:
                y_upper = min(y_upper, floor(boundary + EPSILON))
            else:
                y_lower = max(y_lower, ceil(boundary - EPSILON))
        elif relation == ">=":
            if coefficient_y > 0:
                y_lower = max(y_lower, ceil(boundary - EPSILON))
            else:
                y_upper = min(y_upper, floor(boundary + EPSILON))
        elif relation == "=":
            y_lower = max(y_lower, ceil(boundary - EPSILON))
            y_upper = min(y_upper, floor(boundary + EPSILON))
        else:
            raise ValueError(f"Unsupported relation: {relation}")

        if y_lower > y_upper:
            return None

    return int(y_lower), int(y_upper)


def best_candidate_for_fixed_x(problem, x_value):
    interval = feasible_interval_for_y(problem, x_value)
    if interval is None:
        return None

    y_lower, y_upper = interval
    coefficient_y = problem["objective"][1]

    if problem["sense"] == "max":
        y_value = y_upper if coefficient_y >= 0 else y_lower
    else:
        y_value = y_lower if coefficient_y >= 0 else y_upper

    candidate = (int(x_value), int(y_value))
    return candidate if is_feasible(problem, candidate) else None


def preferred_y_range(problem, y_lower, y_upper):
    coefficient_y = problem["objective"][1]
    should_descend = (
        (problem["sense"] == "max" and coefficient_y >= 0)
        or (problem["sense"] == "min" and coefficient_y < 0)
    )
    if should_descend:
        return range(y_upper, y_lower - 1, -1)
    return range(y_lower, y_upper + 1)


def solve_two_variable_naive(problem):
    best_values = None
    x_lower, x_upper = problem["bounds"][0]
    y_lower, y_upper = problem["bounds"][1]

    for values in product(range(x_lower, x_upper + 1), range(y_lower, y_upper + 1)):
        if is_feasible(problem, values):
            best_values = choose_better(problem, values, best_values)

    if best_values is None:
        raise ValueError("No feasible solution found.")

    return solution_from_values(problem, best_values)


def solve_two_variable_backtracking(problem):
    best_values = None
    x_lower, x_upper = problem["bounds"][0]

    def backtrack(index, current_values):
        nonlocal best_values

        if index == 0:
            for x_value in range(x_lower, x_upper + 1):
                current_values[0] = x_value
                optimistic_values = best_candidate_for_fixed_x(problem, x_value)
                if optimistic_values is None:
                    continue
                if best_values is not None and compare_values(problem, optimistic_values, best_values) <= 0:
                    continue
                backtrack(1, current_values)
            return

        interval = feasible_interval_for_y(problem, current_values[0])
        if interval is None:
            return

        y_lower, y_upper = interval
        for y_value in preferred_y_range(problem, y_lower, y_upper):
            current_values[1] = y_value
            candidate = tuple(current_values)
            if is_feasible(problem, candidate):
                best_values = choose_better(problem, candidate, best_values)

    backtrack(0, [0, 0])

    if best_values is None:
        raise ValueError("No feasible solution found.")

    return solution_from_values(problem, best_values)


def solve_two_variable_reduced_search(problem):
    best_values = None
    x_lower, x_upper = problem["bounds"][0]

    for x_value in range(x_lower, x_upper + 1):
        candidate = best_candidate_for_fixed_x(problem, x_value)
        if candidate is not None:
            best_values = choose_better(problem, candidate, best_values)

    if best_values is None:
        raise ValueError("No feasible solution found.")

    return solution_from_values(problem, best_values)


def solve_two_variable_dp(problem):
    x_lower, x_upper = problem["bounds"][0]

    @lru_cache(maxsize=None)
    def dp(x_value):
        if x_value > x_upper:
            return None

        current = best_candidate_for_fixed_x(problem, x_value)
        future = dp(x_value + 1)
        return choose_better(problem, current, future)

    best_values = dp(x_lower)
    if best_values is None:
        raise ValueError("No feasible solution found.")

    return solution_from_values(problem, best_values)


def solve_bundle_with_method(problem_bundle, solver_function):
    results = {}
    total_cost = 0

    for species_name, species_problem in problem_bundle.items():
        species_result = solver_function(species_problem)
        results[species_name] = species_result
        total_cost += species_result[species_problem["objective_key"]]

    results["total_cost"] = int(total_cost)
    return results


In [19]:
def solve_linear_system(matrix, rhs):
    size = len(rhs)
    augmented = [row[:] + [rhs_value] for row, rhs_value in zip(matrix, rhs)]

    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row: abs(augmented[row][col]))
        if abs(augmented[pivot_row][col]) < EPSILON:
            return None

        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot = augmented[col][col]

        for j in range(col, size + 1):
            augmented[col][j] /= pivot

        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[row][size] for row in range(size)]


def to_leq_constraints(problem):
    constraints = []
    for coefficients, relation, rhs in problem["constraints"]:
        if relation == "<=":
            constraints.append((list(coefficients), rhs))
        elif relation == ">=":
            constraints.append(([-coefficient for coefficient in coefficients], -rhs))
        elif relation == "=":
            constraints.append((list(coefficients), rhs))
            constraints.append(([-coefficient for coefficient in coefficients], -rhs))
        else:
            raise ValueError(f"Unsupported relation: {relation}")
    return constraints


def solve_lp_relaxation_by_vertices(objective, constraints, bounds, sense):
    number_of_variables = len(objective)
    candidate_equalities = []

    for coefficients, rhs in constraints:
        candidate_equalities.append((coefficients[:], rhs))

    for index, (lower_bound, upper_bound) in enumerate(bounds):
        upper_coefficients = [0.0] * number_of_variables
        upper_coefficients[index] = 1.0
        candidate_equalities.append((upper_coefficients, float(upper_bound)))

        lower_coefficients = [0.0] * number_of_variables
        lower_coefficients[index] = -1.0
        candidate_equalities.append((lower_coefficients, -float(lower_bound)))

    best_solution = None
    best_value = float("-inf") if sense == "max" else float("inf")

    for equality_group in combinations(candidate_equalities, number_of_variables):
        matrix = [list(coefficients) for coefficients, _ in equality_group]
        rhs = [rhs_value for _, rhs_value in equality_group]
        candidate = solve_linear_system(matrix, rhs)

        if candidate is None:
            continue

        if any(
            value < lower_bound - EPSILON or value > upper_bound + EPSILON
            for value, (lower_bound, upper_bound) in zip(candidate, bounds)
        ):
            continue

        if any(dot_product(coefficients, candidate) > rhs_value + EPSILON for coefficients, rhs_value in constraints):
            continue

        candidate_value = dot_product(objective, candidate)
        if sense == "max":
            if candidate_value > best_value + EPSILON:
                best_solution = candidate
                best_value = candidate_value
        else:
            if candidate_value < best_value - EPSILON:
                best_solution = candidate
                best_value = candidate_value

    return best_solution, best_value


def solve_two_variable_branch_and_bound(problem):
    transformed_constraints = to_leq_constraints(problem)
    root_bounds = [(float(lower), float(upper)) for lower, upper in problem["bounds"]]
    best_values = None
    best_objective = float("-inf") if problem["sense"] == "max" else float("inf")
    stack = [root_bounds]

    while stack:
        bounds = stack.pop()
        lp_solution, lp_value = solve_lp_relaxation_by_vertices(
            objective=problem["objective"],
            constraints=transformed_constraints,
            bounds=bounds,
            sense=problem["sense"],
        )

        if lp_solution is None:
            continue

        if best_values is not None:
            if problem["sense"] == "max" and lp_value <= best_objective + EPSILON:
                continue
            if problem["sense"] == "min" and lp_value >= best_objective - EPSILON:
                continue

        fractional_index = None
        for index, value in enumerate(lp_solution):
            if not is_integer_value(value):
                fractional_index = index
                break

        if fractional_index is None:
            integer_values = tuple(int(round(value)) for value in lp_solution)
            if is_feasible(problem, integer_values):
                best_values = choose_better(problem, integer_values, best_values)
                best_objective = objective_value(problem, best_values)
            continue

        fractional_value = lp_solution[fractional_index]
        lower_branch_upper = floor(fractional_value)
        upper_branch_lower = ceil(fractional_value)

        left_bounds = [tuple(bound) for bound in bounds]
        if left_bounds[fractional_index][0] <= lower_branch_upper:
            left_bounds[fractional_index] = (left_bounds[fractional_index][0], float(lower_branch_upper))
            stack.append(left_bounds)

        right_bounds = [tuple(bound) for bound in bounds]
        if upper_branch_lower <= right_bounds[fractional_index][1]:
            right_bounds[fractional_index] = (float(upper_branch_lower), right_bounds[fractional_index][1])
            stack.append(right_bounds)

    if best_values is None:
        raise ValueError("No feasible solution found.")

    return solution_from_values(problem, best_values)


## Problem: Refrigerated Vehicles

Variables:
- `type_a_trucks`
- `type_b_trucks`

Objective:
- minimize rental cost

Capacity constraints:
- refrigerated capacity
- non-refrigerated capacity


In [20]:
VEHICLE_PROBLEM = {
    "objective": [30, 40],
    "constraints": [
        ([20, 30], ">=", 900),
        ([40, 30], ">=", 1200),
    ],
    "bounds": [(0, 45), (0, 40)],
    "variable_names": ["type_a_trucks", "type_b_trucks"],
    "sense": "min",
    "objective_key": "cost",
}

VEHICLE_EXPECTED = {
    "type_a_trucks": 15,
    "type_b_trucks": 20,
    "cost": 1250,
}


In [21]:
@timed
def solve_refrigerated_vehicles_naive():
    return solve_two_variable_naive(VEHICLE_PROBLEM)


In [22]:
@timed
def solve_refrigerated_vehicles_backtracking():
    return solve_two_variable_backtracking(VEHICLE_PROBLEM)


In [23]:
@timed
def solve_refrigerated_vehicles_reduced_search():
    return solve_two_variable_reduced_search(VEHICLE_PROBLEM)


In [24]:
@timed
def solve_refrigerated_vehicles_dp():
    return solve_two_variable_dp(VEHICLE_PROBLEM)


In [25]:
@timed
def solve_refrigerated_vehicles_branch_and_bound():
    return solve_two_variable_branch_and_bound(VEHICLE_PROBLEM)


In [26]:
vehicle_naive_result, vehicle_naive_time = solve_refrigerated_vehicles_naive()
vehicle_backtracking_result, vehicle_backtracking_time = solve_refrigerated_vehicles_backtracking()
vehicle_reduced_search_result, vehicle_reduced_search_time = solve_refrigerated_vehicles_reduced_search()
vehicle_dp_result, vehicle_dp_time = solve_refrigerated_vehicles_dp()
vehicle_branch_and_bound_result, vehicle_branch_and_bound_time = solve_refrigerated_vehicles_branch_and_bound()

print("=== REFRIGERATED VEHICLES RESULTS ===")
print(f"Naive enumeration            -> {vehicle_naive_result}, time = {vehicle_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {vehicle_backtracking_result}, time = {vehicle_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {vehicle_reduced_search_result}, time = {vehicle_reduced_search_time:.8f} seconds")
print(f"Dynamic programming          -> {vehicle_dp_result}, time = {vehicle_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {vehicle_branch_and_bound_result}, time = {vehicle_branch_and_bound_time:.8f} seconds")

assert vehicle_naive_result == VEHICLE_EXPECTED
assert vehicle_backtracking_result == vehicle_naive_result
assert vehicle_reduced_search_result == vehicle_naive_result
assert vehicle_dp_result == vehicle_naive_result
assert vehicle_branch_and_bound_result == vehicle_naive_result
print("All five exact methods agree with the expected result.")


=== REFRIGERATED VEHICLES RESULTS ===
Naive enumeration            -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}, time = 0.00204929 seconds
Backtracking with pruning    -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}, time = 0.00041196 seconds
Constraint-driven reduced search -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}, time = 0.00011817 seconds
Dynamic programming          -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}, time = 0.00011796 seconds
Branch and Bound             -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}, time = 0.00007546 seconds
All five exact methods agree with the expected result.


## Variant Note

Unlike sections `2.2` and `2.3`, the source text does not include a separate “Variación propuesta para el estudiante” for section `2.4`.
